# VELNOR Qwen3-8B QLoRA — general curriculum, single epoch

First real training pass over the **full 288-example curriculum** (138 embedded + 150 external general-curriculum rows) with **one epoch and early stopping**.

Why this configuration: the 20261113 continuation showed validation loss bottoming at epoch 1 and rising afterwards (1.16826 -> 1.17665 -> 1.17989). Three epochs on this dataset size overfits. Fresh adapter, no resume, one epoch, patience 1.

Free GPU only. Preflight requires >= 14 GiB total and >= 12 GiB free VRAM, otherwise nothing starts.


In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError('No CUDA GPU attached. Select a free GPU runtime; CPU training is disabled.')
p = torch.cuda.get_device_properties(0)
free, total = torch.cuda.mem_get_info(0)
print('GPU:', p.name, '| total GiB:', round(total/1024**3,2), '| free GiB:', round(free/1024**3,2))
if total < 14*1024**3 or free < 12*1024**3:
    raise RuntimeError('GPU does not meet the Qwen3-8B QLoRA preflight. No training started.')


In [ ]:
import subprocess
REPO = '/content/velnor-general'
subprocess.run(['rm', '-rf', REPO])
subprocess.run(['git', 'clone', '--branch', 'feat/qwen3-8b-qlora', '--depth', '1',
                'https://github.com/vectolith/velnor.git', REPO], check=True)
print(subprocess.check_output(['git', '-C', REPO, 'rev-parse', 'HEAD'], text=True).strip())


In [ ]:
%cd /content/velnor-general
!python -m pip install -q -r training/requirements-qwen3-qlora.txt
import torch
assert torch.cuda.is_available(), 'CUDA was lost during dependency installation.'
print('CUDA:', torch.version.cuda, '| PyTorch:', torch.__version__)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import os, subprocess
RUNS = Path('/content/drive/MyDrive/VELNOR-training-runs')
RUNS.mkdir(parents=True, exist_ok=True)
run_name = 'qwen3-8b-general-e1-' + subprocess.check_output(['date','-u','+%Y%m%dT%H%M%SZ'], text=True).strip()
RUN_DIR = RUNS / run_name
OUT_DIR = RUN_DIR / 'outputs'
OUT_DIR.mkdir(parents=True, exist_ok=False)
os.environ['VELNOR_QWEN3_OUTPUT'] = str(OUT_DIR)
os.environ.pop('VELNOR_QWEN3_RESUME', None)
os.environ.pop('VELNOR_QWEN3_PREV_BEST_LOSS', None)
os.environ['VELNOR_QWEN3_MAX_SEQ_LEN'] = '256'
os.environ['VELNOR_QWEN3_EPOCHS'] = '1'
os.environ['VELNOR_QWEN3_GRAD_ACCUM'] = '8'
os.environ['VELNOR_QWEN3_PATIENCE'] = '1'
print('run dir:', RUN_DIR)


In [ ]:
%%bash
set -euo pipefail
cd /content/velnor-general/training
python -u train_qwen3_8b_qlora.py 2>&1 | tee "$VELNOR_QWEN3_OUTPUT/training.log"
test -s "$VELNOR_QWEN3_OUTPUT/adapter/adapter_config.json"
test -s "$VELNOR_QWEN3_OUTPUT/training_report.json"
echo TRAINING_DONE


In [ ]:
import json
report = json.loads((OUT_DIR / 'training_report.json').read_text(encoding='utf-8'))
print('status:', report['status'])
print('dataset:', report['dataset']['train_examples'], 'train /', report['dataset']['heldout_examples'], 'heldout')
print('method:', report['method']['epochs_run'], 'epoch(s), lr', report['method']['learning_rate'], 'patience', report['method']['early_stopping_patience'])
print('validation:', report['validation']['pre_training_loss'], '->', report['validation']['best_loss'], 'best epoch', report['validation']['best_epoch'])
print('rubric base:', report['rubric']['base']['passed'], '/', report['rubric']['base']['total'])
print('rubric tuned:', report['rubric']['tuned']['passed'], '/', report['rubric']['tuned']['total'], 'delta', report['rubric']['delta_passed'])
print('promotion gate:', report['promotion_gate'])


In [ ]:
import shutil
archive = shutil.make_archive(str(RUN_DIR / 'velnor-qwen3-8b-general-e1-artifacts'), 'zip', root_dir=str(OUT_DIR))
print('archived:', archive)
print(RUN_DIR)
